In [1]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

print("NEO4J_URI:", os.getenv("NEO4J_URI"))
print("NEO4J_USERNAME:", os.getenv("NEO4J_USERNAME"))
print("NEO4J_PASSWORD:", os.getenv("NEO4J_PASSWORD"))
print("NEO4J_DATABASE:", os.getenv("NEO4J_DATABASE"))

NEO4J_URI: neo4j://127.0.0.1:7687
NEO4J_USERNAME: neo4j
NEO4J_PASSWORD: 12345678
NEO4J_DATABASE: example


In [2]:
# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph()
graph.query("RETURN 'hello'")

Unable to retrieve routing information


ValueError: Could not connect to Neo4j database. Please ensure that the url is correct

In [3]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


# 데이터 생성

## 데이터 모델 구조

소셜 네트워크 데이터 모델을 사용하여 실습

- 노드 레이블:
  - `Person`: 사용자 정보 
    - Property: name, age, gender
  - `Interest`: 관심사. 영화, 음악, 스포츠 등
    - Property: name
  - `Location`: 장소
    - Property: name
  - `City`: 도시
    - property: name, population
  
- 관계 유형:
  - `KNOWS`: Person-Person 간의 관계로 사람간에 서로 알고 지냄을 표현.
    - Property: since - 언제 부터 알고 지냈는지
  - `LIKES`: Person-Interest 간의 관계로 사람이 좋아하는 관심사를 표현
    - Property: rating - 관심이 있는 정도
  - `VISITED`: Person-Location 간의 관계로 사람이 방문장소를 표현
    - Property: date-방문일자, rating- 그 장소를 좋아하는 정도
  - `LIVES_IN`: Person-City간의 관계로 사람이 사는 곳을 표현
    - Property: since - 언제부터 살고 있는지
  - `LOCATED_IN`: Location-City간의 관계로 장소가 어느 도시에 위치하는지를 표현


In [4]:
constraints = [
    # Person 중복 방지 (이름 + 나이 복합키)
    """
    CREATE CONSTRAINT constraint_person_name_age IF NOT EXISTS
    FOR (p:Person)
    REQUIRE (p.name, p.age) IS UNIQUE
    """,
    # Interest 중복 방지
    """
    CREATE CONSTRAINT constraint_interest_name IF NOT EXISTS
    FOR (i:Interest)
    REQUIRE i.name IS UNIQUE
    """,
    # City 중복 방지
    """
    CREATE CONSTRAINT constraint_city_name IF NOT EXISTS
    FOR (c:City)
    REQUIRE c.name IS UNIQUE
    """,
    # Location 중복 방지
    """
    CREATE CONSTRAINT constraint_location_name IF NOT EXISTS
    FOR (l:Location)
    REQUIRE l.name IS UNIQUE
    """,
]

for constraint in constraints:
    graph.query(constraint)

print("제약조건 생성 완료")

제약조건 생성 완료


In [5]:
graph.query("SHOW CONSTRAINT")

[{'id': 5,
  'name': 'constraint_city_name',
  'type': 'NODE_PROPERTY_UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['City'],
  'properties': ['name'],
  'enforcedLabel': None,
  'ownedIndex': 'constraint_city_name',
  'propertyType': None},
 {'id': 3,
  'name': 'constraint_interest_name',
  'type': 'NODE_PROPERTY_UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Interest'],
  'properties': ['name'],
  'enforcedLabel': None,
  'ownedIndex': 'constraint_interest_name',
  'propertyType': None},
 {'id': 7,
  'name': 'constraint_location_name',
  'type': 'NODE_PROPERTY_UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Location'],
  'properties': ['name'],
  'enforcedLabel': None,
  'ownedIndex': 'constraint_location_name',
  'propertyType': None},
 {'id': 0,
  'name': 'constraint_person_name_age',
  'type': 'NODE_PROPERTY_UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Person'],
  'properties': ['name', 'age'],
  'enforcedLabel': None,
  'ownedIndex': 'c

In [6]:
# 데이터 모델 정의 
cypher_query =  """
// Person 노드 생성
CREATE (p1:Person {name: '홍길동', age: 35, gender:'남성'})
CREATE (p2:Person {name: '김철수', age: 28, gender:'남성'})
CREATE (p3:Person {name: '이영희', age: 32, gender:'여성'})
CREATE (p4:Person {name: '박지성', age: 40, gender:'남성'})
CREATE (p5:Person {name: '최민수', age: 45, gender:'남성'})
CREATE (p6:Person {name: '정소라', age: 29, gender:'여성'})
CREATE (p7:Person {name: '강대호', age: 33, gender:'남성'})
CREATE (p8:Person {name: '윤미래', age: 27, gender:'여성'})

// 관심분야 노드 생성
CREATE (i1:Interest {name: '영화'})
CREATE (i2:Interest {name: '음악'})
CREATE (i3:Interest {name: '스포츠'})
CREATE (i4:Interest {name: '요리'})
CREATE (i5:Interest {name: '여행'})

// 도시 노드 생성
CREATE (c1:City {name: '서울', population: 9390000})
CREATE (c2:City {name: '인천', population: 2950000})
CREATE (c3:City {name: '부산', population: 3260000})
CREATE (c4:City {name: '대전', population: 1490000})
CREATE (c5:City {name: '광주', population: 1460000})
CREATE (c6:City {name: '대구', population: 2350000})

// 장소 노드 생성
CREATE (l1:Location {name: '남산'})
CREATE (l2:Location {name: '해운대'})
CREATE (l3:Location {name: '경복궁'})
CREATE (l4:Location {name: '월미도'})


// KNOWS 관계 생성 (사람들 간의 친구 관계)
CREATE (p1)-[:KNOWS {since: 2010}]->(p2)
CREATE (p1)-[:KNOWS {since: 2015}]->(p3)
CREATE (p1)-[:KNOWS {since: 2012}]->(p7)
CREATE (p2)-[:KNOWS {since: 2018}]->(p6)
CREATE (p3)-[:KNOWS {since: 2017}]->(p4)
CREATE (p3)-[:KNOWS {since: 2020}]->(p5)
CREATE (p4)-[:KNOWS {since: 2019}]->(p5)
CREATE (p6)-[:KNOWS {since: 2016}]->(p8)
CREATE (p7)-[:KNOWS {since: 2014}]->(p3)
CREATE (p7)-[:KNOWS {since: 2021}]->(p8)
CREATE (p8)-[:KNOWS {since: 2019}]->(p4)

// LIKES 관계 생성 (사람과 관심분야 간의 관계)
CREATE (p1)-[:LIKES {rating: 5}]->(i1)
CREATE (p1)-[:LIKES {rating: 4}]->(i5)
CREATE (p2)-[:LIKES {rating: 5}]->(i3)
CREATE (p3)-[:LIKES {rating: 4}]->(i2)
CREATE (p3)-[:LIKES {rating: 5}]->(i4)
CREATE (p3)-[:LIKES {rating: 3}]->(i1)
CREATE (p4)-[:LIKES {rating: 5}]->(i3)
CREATE (p5)-[:LIKES {rating: 3}]->(i1)
CREATE (p5)-[:LIKES {rating: 4}]->(i5)
CREATE (p6)-[:LIKES {rating: 5}]->(i2)
CREATE (p7)-[:LIKES {rating: 4}]->(i4)
CREATE (p8)-[:LIKES {rating: 4}]->(i5)

// VISITED 관계 생성 (사람과 장소 간의 관계)
CREATE (p1)-[:VISITED {date: '2020-03-15', rating: 4}]->(l1)
CREATE (p1)-[:VISITED {date: '2022-05-23', rating: 5}]->(l2)
CREATE (p2)-[:VISITED {date: '2018-04-15', rating: 3}]->(l2)
CREATE (p3)-[:VISITED {date: '2022-06-05', rating: 5}]->(l3)
CREATE (p4)-[:VISITED {date: '2020-07-01', rating: 4}]->(l4)
CREATE (p4)-[:VISITED {date: '2016-02-28', rating: 5}]->(l1)
CREATE (p5)-[:VISITED {date: '2025-08-03', rating: 4}]->(l3)
CREATE (p6)-[:VISITED {date: '2016-01-17', rating: 5}]->(l2)
CREATE (p7)-[:VISITED {date: '2022-09-02', rating: 3}]->(l1)
CREATE (p8)-[:VISITED {date: '2025-10-21', rating: 4}]->(l2)

//LIVES 관계 생성 (사람과 도시간의 관계)
CREATE (p1)-[:LIVES_IN {since:2019}]-> (c1)
CREATE (p2)-[:LIVES_IN {since:2009}]-> (c3)
CREATE (p3)-[:LIVES_IN {since:2022}]-> (c1)
CREATE (p4)-[:LIVES_IN {since:2000}]-> (c2)
CREATE (p5)-[:LIVES_IN {since:2025}]-> (c4)
CREATE (p6)-[:LIVES_IN {since:2000}]-> (c3)
CREATE (p7)-[:LIVES_IN {since:2013}]-> (c1)
CREATE (p8)-[:LIVES_IN {since:2017}]-> (c5)

// LOCATED_IN 관계 생성 (장소가 어느도시에 있는지)
CREATE (l1)-[:LOCATED_IN]-> (c1)
CREATE (l2)-[:LOCATED_IN]-> (c3)
CREATE (l3)-[:LOCATED_IN]-> (c1)
CREATE (l4)-[:LOCATED_IN]-> (c2)
"""

graph.query(cypher_query)

[]

In [7]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Person {name: STRING, age: INTEGER, gender: STRING}
Interest {name: STRING}
City {name: STRING, population: INTEGER}
Location {name: STRING}
Relationship properties:
KNOWS {since: INTEGER}
LIKES {rating: INTEGER}
VISITED {rating: INTEGER, date: STRING}
LIVES_IN {since: INTEGER}
The relationships:
(:Person)-[:KNOWS]->(:Person)
(:Person)-[:LIKES]->(:Interest)
(:Person)-[:VISITED]->(:Location)
(:Person)-[:LIVES_IN]->(:City)
(:Location)-[:LOCATED_IN]->(:City)


In [ ]:
########################################
# Neo4j Desktop에서 Database를 변경
#
# :use DB이름
# :use example
########################################



### 경로(Path) 쿼리


- 조회하려고 하는 관계의 단계를 지정하여 검색할 수있다.
- `*` 연산자를 사용하여 표현한다
- **핵심 문법:**

    ```cypher
    // 변수 길이 패턴: 1~3 hop
    MATCH (a)-[:REL*1..3]->(b)

    // 정확히 N hop (hop: 관계 단계)
    MATCH (a)-[:REL*2]->(b)

    // 1 이상 (제한 없음 - 주의!)
    MATCH (a)-[:REL*]->(b)

    // 경로 변수
    MATCH path = (a)-[:REL*1..3]->(b)
    RETURN path, length(path), nodes(path), relationships(path);

    //length(path): 경로에 포함된 관계의 개수, 즉 hop 수를 반환
    //nodes(path): 경로에 포함된 노드들을 리스트로 반환
    //relationships(path): 경로에 포함된 관계들을 리스트로 반환

    // 최단 경로 조회
    MATCH path = shortestPath((a)-[:REL*..10]->(b))
    RETURN path;

    // 모든 최단 경로(같은 길이일 때)
    MATCH path = allShortestPaths((a)-[:REL*..10]->(b))
    RETURN path;
    ```


- **예제 1**: 1~3단계 친구 관계 조회

    - 홍길동과 1~3단계 떨어진 모든 사람을 찾아 반환. 
    - 결과는 홍길동의 직접적인 친구(김철수, 이영희, 강대호)뿐만 아니라 친구의 친구, 친구의 친구의 친구까지 포함해서 조회한다.

In [13]:
########################################################################
#  홍길동으로부터 1~3단계 KNOWS 관계로 연결된 모든 사람을 찾는다.
# `*1..3`
########################################################################

cypher_query = """
MATCH (p1:Person{name:"홍길동"})-[r:KNOWS*1..3]->(p2:Person)

RETURN p1.name AS 시작점,[rel IN r | type(rel)+'-'+rel.since]  AS 관계, p2.name AS 도착점
"""
# 여러단계: list[tuple]
# [({}, 'KNOWS', {}), ({}, 'KNOWS', {}), ({}, 'KNOWS', {})]
# type(관계) - 관계의 타입
results = graph.query(cypher_query)
for result in results:
    print(result)
    print("---------------------------------"*4)

{'시작점': '홍길동', '관계': ['KNOWS-2010'], '도착점': '김철수'}
------------------------------------------------------------------------------------------------------------------------------------
{'시작점': '홍길동', '관계': ['KNOWS-2010', 'KNOWS-2018'], '도착점': '정소라'}
------------------------------------------------------------------------------------------------------------------------------------
{'시작점': '홍길동', '관계': ['KNOWS-2010', 'KNOWS-2018', 'KNOWS-2016'], '도착점': '윤미래'}
------------------------------------------------------------------------------------------------------------------------------------
{'시작점': '홍길동', '관계': ['KNOWS-2015'], '도착점': '이영희'}
------------------------------------------------------------------------------------------------------------------------------------
{'시작점': '홍길동', '관계': ['KNOWS-2015', 'KNOWS-2017'], '도착점': '박지성'}
------------------------------------------------------------------------------------------------------------------------------------
{'시작점': '홍길동', '관계': ['K

> ### 컴프리헨션 
> - `[변수 IN 리스트 | 반환할_값]`
> - 리스트에서 각 노드들을 추출해 뽑아 새 배열을 만든다.

- **예제 2**: 정확히 2단계 친구 관계 조회

    - 이 쿼리는 홍길동의 친구의 친구만 반환.

In [17]:
########################################################################
# 홍길동으로부터 정확히 2단계 KNOWS 관계로 연결된 사람들만 조회 (친구의 친구)
# `*2`
########################################################################
cypher_query = """
MATCH (p1:Person {name:'홍길동'}) -[r:KNOWS*2]-> (p2:Person)
RETURN p1.name AS 시작, [rel IN r | type(rel)+"-"+rel.since] AS 관계, p2.name AS 도착
"""

graph.query(cypher_query)

[{'시작': '홍길동', '관계': ['KNOWS-2012', 'KNOWS-2014'], '도착': '이영희'},
 {'시작': '홍길동', '관계': ['KNOWS-2015', 'KNOWS-2017'], '도착': '박지성'},
 {'시작': '홍길동', '관계': ['KNOWS-2015', 'KNOWS-2020'], '도착': '최민수'},
 {'시작': '홍길동', '관계': ['KNOWS-2010', 'KNOWS-2018'], '도착': '정소라'},
 {'시작': '홍길동', '관계': ['KNOWS-2012', 'KNOWS-2021'], '도착': '윤미래'}]

- **예제 3**: 무제한 단계 경로 조회

    - 이 쿼리는 홍길동으로부터 시작해서 KNOWS 관계로 연결된 모든 사람을 반환. 
    - `*`는 단계 제한 없이 모든 경로를 탐색한다.


In [21]:
########################################################################
# 홍길동으로부터 시작해 모든 KNOWS 관계를 따라 연결된 사람들 조회 
# `*`
########################################################################
cypher_query = """
MATCH (p1:Person {name:'홍길동'})-[r:KNOWS*]->(p2:Person)
RETURN p1.name AS 시작, p2.name AS 도착, size(r) AS 단계수 // size(리스트) : 원소개수
"""

graph.query(cypher_query)

[{'시작': '홍길동', '도착': '김철수', '단계수': 1},
 {'시작': '홍길동', '도착': '정소라', '단계수': 2},
 {'시작': '홍길동', '도착': '윤미래', '단계수': 3},
 {'시작': '홍길동', '도착': '박지성', '단계수': 4},
 {'시작': '홍길동', '도착': '최민수', '단계수': 5},
 {'시작': '홍길동', '도착': '이영희', '단계수': 1},
 {'시작': '홍길동', '도착': '박지성', '단계수': 2},
 {'시작': '홍길동', '도착': '최민수', '단계수': 3},
 {'시작': '홍길동', '도착': '최민수', '단계수': 2},
 {'시작': '홍길동', '도착': '강대호', '단계수': 1},
 {'시작': '홍길동', '도착': '이영희', '단계수': 2},
 {'시작': '홍길동', '도착': '박지성', '단계수': 3},
 {'시작': '홍길동', '도착': '최민수', '단계수': 4},
 {'시작': '홍길동', '도착': '최민수', '단계수': 3},
 {'시작': '홍길동', '도착': '윤미래', '단계수': 2},
 {'시작': '홍길동', '도착': '박지성', '단계수': 3},
 {'시작': '홍길동', '도착': '최민수', '단계수': 4}]

#### 최단 경로(Shortest Path) 조회

- `shortestPath()`
  - 두 노드 간의 최단 경로를 찾는다.

- **예제 1**: 두 사람 간의 최단 경로 찾기
- 홍길동에서 최민수까지 갈 수 있는 가장 짧은 경로를 찾아 반환한다. 

In [24]:
######################################################
# Person 노드 홍길동과 최민수 사이의 최단 경로 조회
# shortestPath() 함수: 두 노드 간의 가장 짧은 경로를 찾는다
###################################################### 
cypher_query = """
MATCH path=shortestpath((p1:Person {name:"홍길동"}) -[r*]-> (p2:Person{name:"최민수"}))
RETURN path
"""

graph.query(cypher_query)

[{'path': [{'gender': '남성', 'name': '홍길동', 'age': 35},
   'KNOWS',
   {'gender': '여성', 'name': '이영희', 'age': 32},
   'KNOWS',
   {'gender': '남성', 'name': '최민수', 'age': 45}]}]

- **예제 2**: 특정 관계로만 연결된 최단 경로

    - 

In [26]:
######################################################################
# 이 쿼리는 KNOWS 관계만을 사용하여 홍길동에서 최민수까지의 최단 경로를 찾는다.
######################################################################
cypher_query = """
MATCH p = shortestpath((p1:Person{name:"홍길동"})-[:KNOWS*]->(p2:Person{name:"최민수"}))

RETURN [node IN nodes(p) | node] AS 경로
"""

graph.query(cypher_query)

[{'경로': [{'gender': '남성', 'name': '홍길동', 'age': 35},
   {'gender': '여성', 'name': '이영희', 'age': 32},
   {'gender': '남성', 'name': '최민수', 'age': 45}]}]

- **예제 3**: 경로 길이 제한이 있는 최단 경로

    - 이 쿼리는 최대 3단계의 관계 안에서 홍길동에서 정소라까지의 최단 경로를 조회한다.

In [30]:
#################################################################
# 최대 3단계 관계 내에서 홍길동과 정소라 사이의 최단 경로 조회
#################################################################

cypher_query = """
MATCH p = shortestPath((p1:Person{name:"홍길동"}) -[*..3]-> (p2:Person{name:"최민수"}))
RETURN p
"""

graph.query(cypher_query)

[{'p': [{'gender': '남성', 'name': '홍길동', 'age': 35},
   'KNOWS',
   {'gender': '여성', 'name': '이영희', 'age': 32},
   'KNOWS',
   {'gender': '남성', 'name': '최민수', 'age': 45}]}]

## 집계 함수(Aggregation Functions)

- 여러 행을 하나의 값으로 집계 요약하는 함수. **`RETURN`이나 `WITH` 절에서 사용**한다.
- **주요 집계 함수:**

    | 함수 | 설명 |
    |------|------|
    | `count(x)` | x의 개수 (null 제외) |
    | `count(*)` | 행 개수 |
    | `sum(x)` | 합 |
    | `avg(x)` | 평균 |
    | `min(x)` / `max(x)` | 최솟값 / 최댓값 |
    | `collect(x)` | 리스트로 모으기 |

> -**암묵적 GROUP BY**  
>     - Cypher에는 `GROUP BY`가 없다. **집계 함수와 함께 등장한 다른 컬럼이 자동으로 그룹화 키**가 된다. 
>     - 예: `RETURN d.name, count(m)` → `d.name`별로 그룹핑 후 영화 수 카운트.

In [3]:
#############################
#  전체 Person 노드 수 계산
#############################
cypher_query = """
MATCH (p:Person)
RETURN count(p) AS 전체사람수
"""

graph.query(cypher_query)

NameError: name 'graph' is not defined

In [ ]:
########################
# 다양한 집계
########################
cypher_query = """
MATCH (p:Person)
RETURN sum(p.age) AS 나이합계,
       round(avg(p.age), 2) AS 나이평균,
       round(stdev(p.age), 3) AS 나이표준편차
       min(p.age) AS 최연소
       max(p.age) AS 최고령
"""

graph.query(cypher_query)

- Relationship의 property값을 집계

In [ ]:
############################
# 방문자 평점의 평균점수 집계
############################
cypher_query = """
MATCH (p:Person)-[v:VISITED]->(l:Location)
RETURN p.name AS 방문객,l.name AS 방문객, avg(v.rating) AS 평균관심도 // round(avg(v.rating), 2)
ORDER BY 평균관심도 DESC
"""

graph.query(cypher_query)

In [ ]:
############################################################
#  collect(): 조회결과가 여러개일 때 리스트로 묶어준다.
############################################################
cypher_query = """
MATCH (p:Person)
RETURN collect(p.name) AS 이름
RETURN collect(p)
RETURN collect(p.name) AS 이름들, collect(p.age) AS 나이들
"""

result = graph.query(cypher_query)
result

- 그룹별 집계

In [ ]:
##################################
# 카테고리별 집계
#  성별(gender) Person 노드 수 계산
#  집계함수와 변수를 같이 사용하면 그 변수를 기준으로 자동 그룹화가 이뤄진다.
################################## 
cypher_query = """
MATCH (p:Person)
RETURN p.gender AS 성별, count(p) AS 사람수
ORDER BY 성별//사람수
"""

graph.query(cypher_query)

In [ ]:
##################################
# 관계(Relationship) 속성별 집계
##################################
cypher_query = """
MATCH (p:Person) -[:LIKES]-> (i:Interest)
RETURN i.name AS 관심분야, count(p) AS 사람수
"""

graph.query(cypher_query)

#### 5. WITH 절

- WITH` 는 쿼리를 단계적으로 쪼개서, 앞 단계의 결과를 뒤 단계의 입력으로 넘겨주는 파이프의 역할을 한다.
- **주요 용도:**

  1. 집계 결과를 이후 절에서 추가 필터링/매칭에 사용
     - 집계를 where 절에서 직접 할 수 없다.
     - with 절에 집계함수와 다른 변수가 들어 갈 경우 다른 변수는 group by의 기준이된다.
  2. 중간 결과를 줄이거나 정렬한 뒤 다음 단계 진행
     - return 전에만 limit를 할 경우 많은 양의 조회결과를 끝까지 처리해야 한다. with를 이용해 중간에 데이터의 양을 줄인 뒤 다음을 진행할 수있다.
  3. 변수 범위(scope) 제어
     - with 절에 명시 하지 않은 변수는 다음 단계에서 완전히 사라진다. with 절을 사용해 변수를 줄여가면서 쿼리가 집중해야 할 대상을 좁혀가도록 설계할 때 with절을 사용한다.

- **기본 문법:**

    ```cypher
    MATCH 패턴1
    WITH 변수, 표현식 AS 별칭 [WHERE 조건]
    MATCH 패턴2
    RETURN ...
    ```
- `WITH` 다음에는 `WITH` 절에 등장한 변수만 사용할 수 있다. 이전 변수를 계속 쓰려면 `WITH`에 명시해야 한다.    

- 기본 필터링

In [ ]:
#########################################################
# WITH 기본 작동 원리
# 20세 이상인 사람 중에서 이름에 '김'이 포함된 사람 찾기
#########################################################
cypher_query = """
MATCH (p:Person)
WHERE p.age >= 20 AND CONTAINS '김'
// RETURN p
// 위와 같이 하는게 맞지만 위드절의 개념을 보려고 하단 설명

WITH p  // 위 match에서 찾은 결과 (p)를 다음 구문으로 전달.
WHERE p.name CONTAINS '김' // 김을 포함하고 있는 사람들만 다시 조회 하겠다.
RETURN p.name AS 이름, p.age AS 나이, i.name
"""
graph.query(cypher_query)

- 집계 결과를 중간 처리한 후 다시 활용

In [ ]:
################################################################
# 집계 결과에 대한 조건 필터링 - SQL의 having
# - WITH 문을 사용해야 한다.
# 
# 성별 평균 나이를 계산한 후, 평균 나이가 35세 이상인 성만 반환
################################################################
# 집계함수: RETURN절, WITH절에만 사용 가능.
cypher_query = """
MATCH (p:Person)

RETURN p.gender AS 성별, avg(p.age) AS 나이평균 // {'성별': '남성', '나이평균': 36.2}, {'성별':'여성', '나이평균':29.3333333333333332} 전달
WITH p.gender AS 성별, avg(p.age) AS 나이평균
WHERE 나이평균 ) >= 35
RETURN 나이평균
"""

graph.query(cypher_query)

In [ ]:
####################################################################################
#  도시별 사는 사람들의 평균나이를 계산 한 뒤, 평균나이가 30세 이상인 도시만 반환
####################################################################################

cypher_query = """
MATCH (p:Person) -[:LIVES_IN]-> (c:City)
WITH c.name AS 도시명, avg(p.age) AS 평균나이
WHERE 평균나이 >= 30
RETURN 도시명, 평균나이
"""

graph.query(cypher_query)

- **예제 3**: 복잡한 집계 작업

In [ ]:
#####################################################################################
# 사람별로 방문한 장소 수와 관심분야 수를 계산하여, 둘 다 2개 이상인 사람 찾기
#####################################################################################
cypher_query = """
MATCH (p:Person)
OPTIONAL MATCH (p)-[:VISITED]->(l:Location)

WITH p, count(l) AS 방문장소수
OPTIONAL MATCH (p)-[:LIKES]->(i:Interest)
RETURN p, 방문장소수, count(i) AS 관심분야수

WITH p, 방문장소수>= 2, 관심분야수 >= 2

RETURN 
"""

graph.query(cypher_query)

NameError: name 'graph' is not defined

## 다양한 예제

In [ ]:
# 홍길동과 직접 모르는(KNOWS) 사람들을 조회
cypher_query = """
MATCH (p1:Person {name:'홍길동'}), (p2:Person)
WHERE p1 <> p2
    AND NOT (p1)-[:KNOWS]->(p2) // p1과 p2는 KNOWS관계가 아니다.
RETURN collect(p2.name) AS 모르는사람이름
"""

graph.query(cypher_query)

NameError: name 'graph' is not defined

- 공통 관심사 조회
  - 홍길동과 아는 사이가 아닌 사람 중 공통 관심사를 가지고 있는 사람과 공통관심사 조회

In [ ]:
# 영화에 관심이 있는 사람의 친구의 친구 (영화에 관심있는 사람)

cypher_query = """
MATCH (p1:Person)-[:LIKES]->(i:Interest{name:"영화"})
MATCH (p1)-[:KNOWS*2]->(p2:Person)-[:LIKES]->(i)
RETURN p1.name AS 시작사람, p2.name AS 종료사람, i.name AS 공통관심사항
"""

graph.query(cypher_query)

- 최단 경로와 집계 함수 결합
  - 사람간의 최단 거리를 계산하고 그 거리별 COUNT를 계산.

In [ ]:
cypher_query = """
MATCH p=shortestPath((start:Person{name:'홍길동'})-[:KNOWS*]->(other:Person))
WHERE start <> other
WITH other, length(p) AS 거리단계
RETURN 거리단계, count(other) AS 사람수
ORDER BY 거리단계
"""

graph.query(cypher_query)
